In [0]:
import re
from pyspark.sql import functions as F

def normalize_column_name(column_name):

    column_name = column_name.strip().lower()

    # Replace special characters and spaces
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)

    # Remove leading and trailing underscores
    column_name = column_name.strip("_")

    return column_name


def dynamic_column_mapping(df, column_mapping=None):

    column_mapping = column_mapping or {}

    renamed_columns = []
    final_names = []

    for old_column in df.columns:

        # Normalize source column
        normalized_name = normalize_column_name(old_column)

        # Apply explicit mapping if configured
        final_name = column_mapping.get(
            normalized_name,
            normalized_name
        )

        if final_name in final_names:
            raise ValueError(
                f"Duplicate column mapping detected: {final_name}"
            )

        final_names.append(final_name)

        renamed_columns.append(
            F.col(f"`{old_column}`").alias(final_name)
        )

    return df.select(*renamed_columns)